# Construccion de datasets con comorbilidades

## Objetivo

Este notebook deja documentada y ejecutable la construccion de dos nuevas variantes del dataset final de modelado:

- `multihot_comorb`
- `multihot_abx_comorb`

La regla del proyecto se mantiene: aunque exista un script `.py`, la logica y los resultados importantes deben quedar visibles tambien desde Jupyter Lab.


## Diseño metodologico

No se usa el archivo crudo completo de comorbilidades dentro del modelo porque es demasiado granular y ruidoso. En su lugar, se construye una version reducida `multihot` a partir de un subconjunto curado de grupos clinicos.

Los grupos iniciales seleccionados son:

- falla cardiaca congestiva
- estado de trasplante
- diabetes
- tumor solido sin metastasis
- enfermedad pulmonar cronica
- falla renal
- trastornos pancreaticos
- sinusitis


In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

RUTA_PROYECTO = Path.cwd().resolve().parent
RUTA_MODELADO = RUTA_PROYECTO / "4.MODELADO-Y-VALIDACION"
RUTA_PROCESADOS = RUTA_PROYECTO / "3.DATOS-PROCESADOS"
RUTA_SCRIPT = RUTA_MODELADO / "construir_datasets_comorb.py"

RUTA_COMORB = RUTA_PROCESADOS / "armd_s_aureus_base_modelado_multihot_comorb.csv"
RUTA_ABX_COMORB = RUTA_PROCESADOS / "armd_s_aureus_base_modelado_multihot_abx_comorb.csv"
RUTA_MANIFIESTO_COMORB = RUTA_PROCESADOS / "manifiesto_modelado_multihot_comorb.csv"
RUTA_MANIFIESTO_ABX_COMORB = RUTA_PROCESADOS / "manifiesto_modelado_multihot_abx_comorb.csv"
RUTA_CATALOGO = RUTA_PROCESADOS / "catalogo_grupos_comorbilidades.csv"
RUTA_COBERTURA = RUTA_PROCESADOS / "cobertura_grupos_comorbilidades.csv"

print(RUTA_SCRIPT)


D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\4.MODELADO-Y-VALIDACION\construir_datasets_comorb.py


## Ejecutar construccion

La siguiente celda ejecuta el script real de construccion. Esto evita duplicar logica entre notebook y `.py`, pero deja la evidencia de ejecucion dentro del flujo de trabajo del proyecto.


In [2]:
resultado = subprocess.run(
    [sys.executable, str(RUTA_SCRIPT)],
    cwd=RUTA_MODELADO,
    capture_output=True,
    text=True,
    check=True,
)
print(resultado.stdout)
if resultado.stderr.strip():
    print('STDERR:')
    print(resultado.stderr)


Dataset generado: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\armd_s_aureus_base_modelado_multihot_comorb.csv
Dataset generado: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\armd_s_aureus_base_modelado_multihot_abx_comorb.csv
Manifiesto generado: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\manifiesto_modelado_multihot_comorb.csv
Manifiesto generado: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\manifiesto_modelado_multihot_abx_comorb.csv
Catalogo generado: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\catalogo_grupos_comorbilidades.csv
Cobertura generada: D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\cobertura_grupos_comorbilidades.csv
Columnas multihot comorb: 8



## Catalogo de grupos de comorbilidades

Este catalogo muestra como se mapearon componentes originales del archivo masivo hacia grupos binarios `multihot` mas controlables.

Importante:

- el catalogo no representa columnas finales uno a uno
- puede tener mas filas que grupos distintos
- eso pasa porque un grupo `comorb_*` puede agrupar varios componentes originales


In [3]:
df_catalogo = pd.read_csv(RUTA_CATALOGO)
display(df_catalogo)


,grupo_multihot,comorbidity_component_original
0,comorb_congestive_heart_failure,Congestive heart failure
1,comorb_organ_transplant_status,Organ transplant status
2,comorb_diabetes_any,Diabetes mellitus without complication
3,comorb_diabetes_any,"Diabetes, complicated"
4,comorb_diabetes_any,"Diabetes, uncomplicated"
5,comorb_solid_tumor_non_metastatic,Solid tumor without metastasis
6,comorb_chronic_pulmonary_any,Chronic pulmonary disease
7,comorb_chronic_pulmonary_any,Chronic obstructive pulmonary disease and bron...
8,comorb_renal_failure,Renal failure
9,comorb_pancreatic_disorder,Pancreatic disorders (excluding diabetes)


## Cobertura de los grupos seleccionados

Esta tabla permite ver cuantas ordenes activan cada grupo binario de comorbilidad.

Si el catalogo tiene mas filas que esta tabla, no es error: el catalogo es un mapa componente→grupo, mientras que esta cobertura ya resume por grupo final.


In [4]:
df_cobertura = pd.read_csv(RUTA_COBERTURA)
display(df_cobertura)


,grupo_multihot,ordenes_con_valor_1,porcentaje_ordenes_con_valor_1
0,comorb_congestive_heart_failure,7252,89.65
1,comorb_organ_transplant_status,635,7.85
2,comorb_diabetes_any,2404,29.72
3,comorb_solid_tumor_non_metastatic,724,8.95
4,comorb_chronic_pulmonary_any,2316,28.63
5,comorb_renal_failure,1090,13.48
6,comorb_pancreatic_disorder,1952,24.13
7,comorb_sinusitis,5227,64.62


## Verificacion de datasets generados

Aqui se revisan forma, columnas nuevas y una muestra de cada variante.


In [5]:
df_comorb = pd.read_csv(RUTA_COMORB)
df_abx_comorb = pd.read_csv(RUTA_ABX_COMORB)

print('multihot_comorb shape:', df_comorb.shape)
print('multihot_abx_comorb shape:', df_abx_comorb.shape)

columnas_comorb = [c for c in df_comorb.columns if c.startswith('comorb_')]
columnas_abx = [c for c in df_abx_comorb.columns if c.startswith('exp_prev_')]

print('Columnas multihot comorb:', len(columnas_comorb))
print('Columnas multihot abx:', len(columnas_abx))
print(columnas_comorb)


multihot_comorb shape: (82319, 25)
multihot_abx_comorb shape: (82319, 43)
Columnas multihot comorb: 8
Columnas multihot abx: 18
['comorb_congestive_heart_failure', 'comorb_organ_transplant_status', 'comorb_diabetes_any', 'comorb_solid_tumor_non_metastatic', 'comorb_chronic_pulmonary_any', 'comorb_renal_failure', 'comorb_pancreatic_disorder', 'comorb_sinusitis']


C:\Users\Johan\AppData\Local\Temp\ipykernel_12164\971124935.py:2: DtypeWarning: Columns (0: gender) have mixed types. Specify dtype option on import or set low_memory=False.
  df_abx_comorb = pd.read_csv(RUTA_ABX_COMORB)


In [7]:
display(df_comorb.head())
display(df_abx_comorb.head())


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,antibiotic,culture_description,age,gender,hosp_ward_ICU,hosp_ward_ER,...,median_cr,susceptibility,comorb_congestive_heart_failure,comorb_organ_transplant_status,comorb_diabetes_any,comorb_solid_tumor_non_metastatic,comorb_chronic_pulmonary_any,comorb_renal_failure,comorb_pancreatic_disorder,comorb_sinusitis
0,JC1245002,131309498310,722915926,2021-05-23 17:17:00+00:00,Linezolid,RESPIRATORY,55-64 years,1,0,0,...,NaN,Susceptible,0,0,0,0,0,0,0,1
1,JC2474951,131237027276,532673537,2017-08-29 17:54:00+00:00,Linezolid,RESPIRATORY,18-24 years,0,0,0,...,NaN,Susceptible,1,0,0,0,1,0,0,1
2,JC2790465,131266973373,607899842,2019-04-18 17:28:00+00:00,Linezolid,RESPIRATORY,65-74 years,1,1,0,...,0.97,Susceptible,1,0,0,0,0,0,0,0
3,JC2526401,131262416443,590232155,2019-01-03 03:46:00+00:00,Linezolid,RESPIRATORY,25-34 years,0,0,0,...,NaN,Susceptible,1,0,0,0,0,0,0,1
4,JC696161,131260275836,594498507,2019-02-03 17:50:00+00:00,Linezolid,RESPIRATORY,45-54 years,0,0,0,...,NaN,Susceptible,1,0,0,0,0,0,0,1


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,antibiotic,culture_description,age,gender,hosp_ward_ICU,hosp_ward_ER,...,exp_prev_tetracycline,exp_prev_urinary_antiseptic,comorb_congestive_heart_failure,comorb_organ_transplant_status,comorb_diabetes_any,comorb_solid_tumor_non_metastatic,comorb_chronic_pulmonary_any,comorb_renal_failure,comorb_pancreatic_disorder,comorb_sinusitis
0,JC1245002,131309498310,722915926,2021-05-23 17:17:00+00:00,Linezolid,RESPIRATORY,55-64 years,1,0,0,...,0,0,0,0,0,0,0,0,0,1
1,JC2474951,131237027276,532673537,2017-08-29 17:54:00+00:00,Linezolid,RESPIRATORY,18-24 years,0,0,0,...,0,0,1,0,0,0,1,0,0,1
2,JC2790465,131266973373,607899842,2019-04-18 17:28:00+00:00,Linezolid,RESPIRATORY,65-74 years,1,1,0,...,0,0,1,0,0,0,0,0,0,0
3,JC2526401,131262416443,590232155,2019-01-03 03:46:00+00:00,Linezolid,RESPIRATORY,25-34 years,0,0,0,...,0,0,1,0,0,0,0,0,0,1
4,JC696161,131260275836,594498507,2019-02-03 17:50:00+00:00,Linezolid,RESPIRATORY,45-54 years,0,0,0,...,0,0,1,0,0,0,0,0,0,1


## Manifiestos de modelado

Estos manifiestos aclaran que entra como predictor y que rol cumple cada columna.


In [8]:
display(pd.read_csv(RUTA_MANIFIESTO_COMORB))
display(pd.read_csv(RUTA_MANIFIESTO_ABX_COMORB))


,variable,rol,entra_como_predictor
0,anon_id,identificador,0
1,pat_enc_csn_id_coded,identificador,0
2,order_proc_id_coded,identificador,0
3,order_time_jittered_utc,temporal_para_split,0
4,antibiotic,predictor_base,1
5,culture_description,predictor_base,1
6,age,predictor_base,1
7,gender,predictor_base,1
8,hosp_ward_ICU,predictor_base,1
9,hosp_ward_ER,predictor_base,1


,variable,rol,entra_como_predictor
0,anon_id,identificador,0
1,pat_enc_csn_id_coded,identificador,0
2,order_proc_id_coded,identificador,0
3,order_time_jittered_utc,temporal_para_split,0
4,antibiotic,predictor_base,1
5,culture_description,predictor_base,1
6,age,predictor_base,1
7,gender,predictor_base,1
8,hosp_ward_ICU,predictor_base,1
9,hosp_ward_ER,predictor_base,1


## Las 5 bases candidatas para entrenar o comparar modelos

Despues de esta construccion, el proyecto queda con `5` datasets candidatos:

1. `armd_s_aureus_base_modelado_base.csv`
2. `armd_s_aureus_base_modelado_ampliada.csv`
3. `armd_s_aureus_base_modelado_multihot_abx.csv`
4. `armd_s_aureus_base_modelado_multihot_comorb.csv`
5. `armd_s_aureus_base_modelado_multihot_abx_comorb.csv`

La idea metodologica es compararlos, no asumir desde ya que la version mas grande sera automaticamente la mejor.
